# Benchmark de Random Forest: scikit-learn, Bankai e LightGBM

Este notebook compara três cargas no mesmo dataset binário sintético: **10.000 linhas, 500 features (90 relevantes)**, split estratificado 80/20 e matriz `float32`. Os parâmetros de árvore do notebook anterior foram mantidos: 40 árvores, profundidade máxima 20, até 511 folhas, mínimo de 5 amostras por folha, bootstrap/subamostragem de 80%, todas as features, seed 42 e `n_jobs=-1`.

1. **Split importance:** scikit-learn RF com contagem de splits extraída das árvores vs Bankai exato (`max_bins=None`), sem aceleração por histogramas.
2. **Permutation importance:** permutation importance do scikit-learn sobre validação vs importance OOB nativa do Bankai exato.
3. **Histogramas:** Bankai com 63 bins vs LightGBM RF com 63 bins, ambos com importance por gain.

Cada variante tem um aquecimento e dez rodadas medidas em ordem alternada. Os resultados aparecem somente em tabelas: medições individuais e medianas por modelo. A avaliação de qualidade (`accuracy`, `precision`, `recall`, `F1` e `ROC-AUC`) e as chamadas de `predict_proba` ficam fora dos cronômetros. A permutation importance OOB nativa do Bankai é calculada dentro de `fit`; no sklearn ela é uma etapa externa, então a tabela também mostra o tempo de `fit` mais importance externa para essa comparação.

A primeira célula instala as dependências para Google Colab.

In [8]:
%pip install --upgrade -q "bankai-random-forest" "scikit-learn==1.9.1" "lightgbm==4.7.0"

/Users/jesus/Documents/Code/bankai-random-forest-v2/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 1. Runtime e dependências

Confira versões e CPUs disponíveis antes de iniciar as medições.

In [9]:
import gc
import os
import platform
import time
from importlib.metadata import version

import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from joblib import cpu_count
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from bankai_random_forest import BankaiRandomForestClassifier

print(f"Python: {platform.python_version()}")
print(f"Bankai: {version('bankai-random-forest')}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Logical CPUs: {os.cpu_count()}; joblib CPUs: {cpu_count()}")

Python: 3.11.11
Bankai: 0.1.0a7
scikit-learn: 1.9.1
LightGBM: 4.7.0
NumPy: 2.4.6
Logical CPUs: 8; joblib CPUs: 8


## 2. Dataset comum de 10 mil linhas

Geramos uma matriz uma única vez e compartilhamos os mesmos arrays entre os modelos. Os 90 primeiros atributos carregam sinal; os outros 410 são ruído.

In [10]:
N_SAMPLES = 10_000
N_FEATURES = 500
N_RELEVANT = 90
DATA_SEED = 1789

rng = np.random.default_rng(DATA_SEED)
X = rng.standard_normal((N_SAMPLES, N_FEATURES), dtype=np.float32)
coefficients = rng.choice(np.array([-1.0, 1.0], dtype=np.float32), size=N_RELEVANT)
coefficients *= rng.uniform(0.5, 1.5, size=N_RELEVANT).astype(np.float32)
signal = X[:, :N_RELEVANT] @ coefficients
signal /= np.sqrt(np.sum(coefficients ** 2))
probability = 1.0 / (1.0 + np.exp(-signal))
y = rng.binomial(1, probability).astype(np.uint8)
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=0.2, random_state=DATA_SEED, stratify=y
)
assert X.shape == (10_000, 500) and X.dtype == np.float32
print(f"X={X.shape}, dtype={X.dtype}, {X.nbytes / 2**20:.1f} MiB")
print(f"Treino={X_train.shape}; validação={X_validation.shape}; classes={np.bincount(y).tolist()}")
print(f"Features relevantes: 0–{N_RELEVANT - 1}; ruído: {N_RELEVANT}–{N_FEATURES - 1}")

X=(10000, 500), dtype=float32, 19.1 MiB
Treino=(8000, 500); validação=(2000, 500); classes=[5027, 4973]
Features relevantes: 0–89; ruído: 90–499


## 3. Configuração das árvores e executor

Os parâmetros abaixo preservam a configuração do notebook anterior. A importância por split do scikit-learn é calculada contando quantas vezes cada atributo aparece em nós internos das 40 árvores, para corresponder à semântica de contagem de splits do Bankai.

In [11]:
N_JOBS = -1
N_WARMUPS = 1
N_REPEATS = 10
N_ESTIMATORS = 40
MAX_DEPTH = 20
MAX_LEAF_NODES = 511
MIN_SAMPLES_LEAF = 5
SAMPLE_FRACTION = 0.8
MODEL_SEED = 42
MAX_FEATURES = None
MAX_BINS = 63
BIN_SAMPLE_SIZE = 200_000

COMMON_FOREST_PARAMS = dict(
    n_estimators=N_ESTIMATORS, criterion="gini", max_depth=MAX_DEPTH,
    max_leaf_nodes=MAX_LEAF_NODES, min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=MAX_FEATURES, bootstrap=True, max_samples=SAMPLE_FRACTION,
    n_jobs=N_JOBS, random_state=MODEL_SEED,
)

def make_sklearn_rf(n_jobs=N_JOBS):
    return RandomForestClassifier(**{**COMMON_FOREST_PARAMS, "n_jobs": n_jobs})

def make_bankai(max_bins=None, importance_type="split"):
    params = dict(COMMON_FOREST_PARAMS)
    return BankaiRandomForestClassifier(
        **params, max_bins=max_bins, importance_type=importance_type,
        **({"bin_sample_size": BIN_SAMPLE_SIZE} if max_bins is not None else {}),
    )

def sklearn_split_importance(model):
    counts = np.zeros(N_FEATURES, dtype=np.float64)
    for tree in model.estimators_:
        used = tree.tree_.feature
        used = used[used >= 0]
        counts += np.bincount(used, minlength=N_FEATURES)
    total = counts.sum()
    return counts / total if total else counts

def run_benchmark(factories, importance_fns, native_importance=(), baseline=None):
    """Measure fit, importance, predict and quality; display per-run and median tables."""
    rows = []
    names = list(factories)
    native_importance = set(native_importance)
    for run in range(-N_WARMUPS, N_REPEATS):
        order = names if (run + N_WARMUPS) % 2 == 0 else list(reversed(names))
        for name in order:
            model = factories[name]()
            gc.collect()
            t0 = time.perf_counter(); model.fit(X_train, y_train); fit_s = time.perf_counter() - t0
            t0 = time.perf_counter(); importance = importance_fns[name](model); importance_s = time.perf_counter() - t0
            t0 = time.perf_counter(); predictions = model.predict(X_validation); predict_s = time.perf_counter() - t0
            # Probability inference and metric calculations are excluded from benchmark timings.
            probabilities = model.predict_proba(X_validation)
            positive_index = int(np.flatnonzero(model.classes_ == 1)[0])
            positive_scores = probabilities[:, positive_index]
            metrics = {
                "accuracy": accuracy_score(y_validation, predictions),
                "precision": precision_score(y_validation, predictions, zero_division=0),
                "recall": recall_score(y_validation, predictions, zero_division=0),
                "f1": f1_score(y_validation, predictions, zero_division=0),
                "roc_auc": roc_auc_score(y_validation, positive_scores),
            }
            assert len(importance) == N_FEATURES and len(predictions) == len(y_validation)
            if run >= 0:
                rows.append({"model": name, "run": run + 1, "fit_seconds": fit_s,
                             "importance_seconds": importance_s, "predict_seconds": predict_s,
                             "importance_in_fit": name in native_importance,
                             "fit_plus_importance_seconds": fit_s + (0 if name in native_importance else importance_s),
                             **metrics})
            del model, importance, predictions, probabilities, positive_scores
    samples = pd.DataFrame(rows)
    median_columns = ["fit_seconds", "importance_seconds", "predict_seconds",
                      "fit_plus_importance_seconds", "accuracy", "precision", "recall", "f1", "roc_auc"]
    medians = samples.groupby("model")[median_columns].median()
    if baseline:
        base = medians.loc[baseline]
        for metric in ("fit_seconds", "importance_seconds", "predict_seconds", "fit_plus_importance_seconds"):
            medians[metric.replace("_seconds", "_speedup_vs_baseline")] = base[metric] / medians[metric]
    print("Medições individuais (10 rodadas):")
    display(samples.drop(columns="importance_in_fit").round(5))
    print("Medianas por modelo:")
    display(medians.round(5))
    return samples, medians

## 4. Split importance: scikit-learn RF vs Bankai exato

Bankai usa `max_bins=None`, portanto esse cenário não usa aceleração por histogramas. Ambos extraem a contagem de nós de split por feature após o fit. A importância do scikit-learn é contagem normalizada derivada dos nós internos; a importância `split` do Bankai é a importância nativa correspondente.

In [12]:
split_factories = {
    "scikit-learn RF — split count": make_sklearn_rf,
    "Bankai exato — split": lambda: make_bankai(max_bins=None, importance_type="split"),
}
split_importance_fns = {
    "scikit-learn RF — split count": sklearn_split_importance,
    "Bankai exato — split": lambda model: model.feature_importances_,
}
split_samples, split_medians = run_benchmark(
    split_factories, split_importance_fns, baseline="scikit-learn RF — split count"
)

Medições individuais (10 rodadas):


,model,run,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc
0,Bankai exato — split,1,9.37727,0.00000,0.00857,9.37727,0.5615,0.57526,0.45327,0.50703,0.58779
1,scikit-learn RF — split count,1,12.50470,0.00066,0.01430,12.50537,0.5555,0.55321,0.55377,0.55349,0.57855
2,scikit-learn RF — split count,2,13.50976,0.00030,0.01314,13.51006,0.5555,0.55321,0.55377,0.55349,0.57855
3,Bankai exato — split,2,11.17933,0.00000,0.00872,11.17933,0.5615,0.57526,0.45327,0.50703,0.58779
4,Bankai exato — split,3,10.51026,0.00000,0.00675,10.51026,0.5615,0.57526,0.45327,0.50703,0.58779
5,scikit-learn RF — split count,3,13.78267,0.00036,0.01407,13.78303,0.5555,0.55321,0.55377,0.55349,0.57855
6,scikit-learn RF — split count,4,13.62586,0.00052,0.01193,13.62638,0.5555,0.55321,0.55377,0.55349,0.57855
7,Bankai exato — split,4,10.65918,0.00000,0.00828,10.65918,0.5615,0.57526,0.45327,0.50703,0.58779
8,Bankai exato — split,5,11.35230,0.00002,0.00965,11.35231,0.5615,0.57526,0.45327,0.50703,0.58779
9,scikit-learn RF — split count,5,14.19207,0.00043,0.01220,14.19251,0.5555,0.55321,0.55377,0.55349,0.57855


Medianas por modelo:


,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc,fit_speedup_vs_baseline,importance_speedup_vs_baseline,predict_speedup_vs_baseline,fit_plus_importance_speedup_vs_baseline
model,,,,,,,,,,,,,
Bankai exato — split,11.12239,0.00000,0.00867,11.12239,0.5615,0.57526,0.45327,0.50703,0.58779,1.24038,259.95134,1.61844,1.24041
scikit-learn RF — split count,13.79602,0.00042,0.01403,13.79635,0.5555,0.55321,0.55377,0.55349,0.57855,1.00000,1.00000,1.00000,1.00000


## 5. Permutation importance: scikit-learn vs Bankai nativo

No scikit-learn, medimos a permutation importance de acurácia na validação, com cinco permutações por feature. Para evitar paralelismo aninhado, a previsão dentro dessa etapa usa uma thread e as features são distribuídas por `n_jobs=-1`. No Bankai, a importância OOB nativa acontece dentro do `fit`; a leitura de `feature_importances_` depois do treino é apenas acesso ao resultado. Assim, compare `fit_plus_external_importance_median_s` para o custo total de fit + importance: no Bankai o custo da importância já está contabilizado em fit.

In [13]:
PERMUTATION_REPEATS = 5
SKLEARN_PERM_NAME = "scikit-learn RF — permutation (validação)"
BANKAI_PERM_NAME = "Bankai exato — permutation OOB nativa"

def sklearn_permutation(model):
    model.set_params(n_jobs=1)
    result = permutation_importance(
        model, X_validation, y_validation, scoring="accuracy",
        n_repeats=PERMUTATION_REPEATS, random_state=MODEL_SEED, n_jobs=N_JOBS,
    )
    return result.importances_mean

permutation_factories = {
    SKLEARN_PERM_NAME: make_sklearn_rf,
    BANKAI_PERM_NAME: lambda: make_bankai(max_bins=None, importance_type="permutation"),
}
permutation_importance_fns = {
    SKLEARN_PERM_NAME: sklearn_permutation,
    BANKAI_PERM_NAME: lambda model: model.feature_importances_,
}
permutation_samples, permutation_medians = run_benchmark(
    permutation_factories, permutation_importance_fns,
    native_importance={BANKAI_PERM_NAME}, baseline=SKLEARN_PERM_NAME,
)

Medições individuais (10 rodadas):


,model,run,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc
0,Bankai exato — permutation OOB nativa,1,11.88003,0.00001,0.00952,11.88003,0.5615,0.57526,0.45327,0.50703,0.58779
1,scikit-learn RF — permutation (validação),1,14.49655,9.14671,0.01541,23.64326,0.5555,0.55321,0.55377,0.55349,0.57855
2,scikit-learn RF — permutation (validação),2,14.58741,9.16709,0.01567,23.75450,0.5555,0.55321,0.55377,0.55349,0.57855
3,Bankai exato — permutation OOB nativa,2,11.79446,0.00000,0.01058,11.79446,0.5615,0.57526,0.45327,0.50703,0.58779
4,Bankai exato — permutation OOB nativa,3,11.81156,0.00000,0.00991,11.81156,0.5615,0.57526,0.45327,0.50703,0.58779
5,scikit-learn RF — permutation (validação),3,14.32794,9.13191,0.01620,23.45985,0.5555,0.55321,0.55377,0.55349,0.57855
6,scikit-learn RF — permutation (validação),4,14.90567,9.11929,0.01560,24.02496,0.5555,0.55321,0.55377,0.55349,0.57855
7,Bankai exato — permutation OOB nativa,4,11.75559,0.00000,0.01053,11.75559,0.5615,0.57526,0.45327,0.50703,0.58779
8,Bankai exato — permutation OOB nativa,5,12.19139,0.00000,0.00844,12.19139,0.5615,0.57526,0.45327,0.50703,0.58779
9,scikit-learn RF — permutation (validação),5,14.15881,9.46869,0.01200,23.62750,0.5555,0.55321,0.55377,0.55349,0.57855


Medianas por modelo:


,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc,fit_speedup_vs_baseline,importance_speedup_vs_baseline,predict_speedup_vs_baseline,fit_plus_importance_speedup_vs_baseline
model,,,,,,,,,,,,,
Bankai exato — permutation OOB nativa,11.99685,0.00000,0.00945,11.99685,0.5615,0.57526,0.45327,0.50703,0.58779,1.21813,6.129856e+06,1.63162,1.99777
scikit-learn RF — permutation (validação),14.61367,9.44926,0.01542,23.96690,0.5555,0.55321,0.55377,0.55349,0.57855,1.00000,1.000000e+00,1.00000,1.00000


## 6. Histogramas: Bankai vs LightGBM RF

Ambos usam histogramas de 63 bins e importância por ganho (`gain`). Os parâmetros principais das árvores são mantidos; LightGBM RF usa amostragem sem reposição (`bagging_fraction=0.8`, `bagging_freq=1`), enquanto Bankai usa bootstrap. Essa diferença de amostragem e as diferenças internas dos algoritmos devem ser consideradas ao interpretar tempos e importâncias.

In [14]:
LIGHTGBM_RF_PARAMS = dict(
    boosting_type="rf", objective="binary", n_estimators=N_ESTIMATORS,
    max_bin=MAX_BINS, max_depth=MAX_DEPTH, num_leaves=MAX_LEAF_NODES,
    min_child_samples=MIN_SAMPLES_LEAF, bagging_freq=1,
    bagging_fraction=SAMPLE_FRACTION, feature_fraction=1.0,
    subsample_for_bin=BIN_SAMPLE_SIZE, n_jobs=N_JOBS,
    random_state=MODEL_SEED, importance_type="gain", verbosity=-1,
)

def make_lightgbm_rf():
    return LGBMClassifier(**LIGHTGBM_RF_PARAMS)

hist_factories = {
    "Bankai histogramas (63 bins) — gain": lambda: make_bankai(max_bins=MAX_BINS, importance_type="gain"),
    "LightGBM RF (63 bins) — gain": make_lightgbm_rf,
}
hist_importance_fns = {
    "Bankai histogramas (63 bins) — gain": lambda model: model.feature_importances_,
    "LightGBM RF (63 bins) — gain": lambda model: model.feature_importances_,
}
hist_samples, hist_medians = run_benchmark(
    hist_factories, hist_importance_fns, baseline="LightGBM RF (63 bins) — gain"
)

Medições individuais (10 rodadas):


,model,run,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc
0,LightGBM RF (63 bins) — gain,1,4.69315,0.00006,0.00561,4.69321,0.570,0.56853,0.56281,0.56566,0.59361
1,Bankai histogramas (63 bins) — gain,1,2.21542,0.00002,0.03430,2.21544,0.554,0.56212,0.46834,0.51096,0.57696
2,Bankai histogramas (63 bins) — gain,2,1.99553,0.00000,0.02684,1.99553,0.554,0.56212,0.46834,0.51096,0.57696
3,LightGBM RF (63 bins) — gain,2,4.30007,0.00005,0.00416,4.30013,0.570,0.56853,0.56281,0.56566,0.59361
4,LightGBM RF (63 bins) — gain,3,4.16137,0.00005,0.00518,4.16142,0.570,0.56853,0.56281,0.56566,0.59361
5,Bankai histogramas (63 bins) — gain,3,1.91437,0.00000,0.02698,1.91437,0.554,0.56212,0.46834,0.51096,0.57696
6,Bankai histogramas (63 bins) — gain,4,1.82842,0.00000,0.02426,1.82842,0.554,0.56212,0.46834,0.51096,0.57696
7,LightGBM RF (63 bins) — gain,4,4.11099,0.00005,0.00408,4.11104,0.570,0.56853,0.56281,0.56566,0.59361
8,LightGBM RF (63 bins) — gain,5,4.20315,0.00005,0.00525,4.20321,0.570,0.56853,0.56281,0.56566,0.59361
9,Bankai histogramas (63 bins) — gain,5,1.98451,0.00001,0.03055,1.98452,0.554,0.56212,0.46834,0.51096,0.57696


Medianas por modelo:


,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc,fit_speedup_vs_baseline,importance_speedup_vs_baseline,predict_speedup_vs_baseline,fit_plus_importance_speedup_vs_baseline
model,,,,,,,,,,,,,
Bankai histogramas (63 bins) — gain,1.90935,0.00000,0.02524,1.90935,0.554,0.56212,0.46834,0.51096,0.57696,2.25033,25.94775,0.19775,2.25035
LightGBM RF (63 bins) — gain,4.29667,0.00005,0.00499,4.29672,0.570,0.56853,0.56281,0.56566,0.59361,1.00000,1.00000,1.00000,1.00000


## Leitura das tabelas

Em cada cenário, a primeira tabela contém os valores de cada rodada medida; a segunda contém as medianas por modelo. Os tempos aparecem em segundos. A tabela de medianas também mostra o speedup contra o modelo de referência para `fit`, importance, `predict` e fit mais importance externa.

`accuracy`, `precision`, `recall`, `f1` e `roc_auc` são calculados na partição de validação e não entram nos tempos cronometrados. `roc_auc` usa as probabilidades da classe positiva. A importance OOB nativa do Bankai é calculada durante o `fit`; por isso `fit_plus_importance_seconds` não soma novamente a duração de leitura do resultado nativo.

O benchmark executa um aquecimento e dez rodadas medidas. Os resultados dependem do hardware e runtime exibidos no início.